In [13]:
import re
import os
from pathlib import Path
import numpy as np

# ─── User‐set variables ──────────────────────────────────────────────────────────
# base_dir      = Path("results")  # ← change this to your root folder
# dataset_name  = "bace"
# split         = "random"
# fewshot_bool  = "False"     # or "False"
# fewshot_num   = "50"       # e.g. "5", "10", "20", ...
# full_or_LP    = "fullFT"     # or "LP"
# loss_fun      = "filter_bce+feature_map"      # e.g. "mse", "cross_entropy", ...

def calc_score(dataset_name, split, fewshot_bool, fewshot_num, full_or_LP, loss_fun):
    base_dir      = Path("results")
    # If you need the “best” folder logic to depend on the dataset:
    LOWER_IS_BETTER = ['lipo', 'esol', 'malaria', 'cep']

    # ─── Build a simple glob‐style pattern ─────────────────────────────────────────
    # Matches e.g. FT_lipo_split1_Fewshot_True_10_full_type_mse_ablationX_lr0.001
    pattern = (
        f"FT_{dataset_name}_{split}_Fewshot_{fewshot_bool}_"
        f"{fewshot_num}_{full_or_LP}_type_{loss_fun}_*"
    )

    # Find all matching top‐level folders
    candidate_dirs = list(base_dir.glob(pattern))
    # print("Pattern matches:     ", [p.name for p in base_dir.glob(pattern)])

    results = {}  # folder → (mean, std)
    for folder in candidate_dirs:
        if not folder.is_dir():
            continue

        # each folder may contain 1–5 subfolders (separate random seeds / runs)
        subdirs = [d for d in folder.iterdir() if d.is_dir()]
        vals = []

        for sd in subdirs:
            log_path = sd / "logging.log"
            if not log_path.exists():
                continue

            lines = log_path.read_text().splitlines()
            marker_idxs = [i for i, l in enumerate(lines) if l.startswith("Avg time per epoch:")]
            if not marker_idxs:
                # no marker → skip this run
                continue

            # take the last one
            last_idx = marker_idxs[-1]
            if last_idx == 0:
                # no previous line to extract from → skip
                continue

            target_line = lines[last_idx - 1].strip()

            # 3) extract the last numeric token
            nums = re.findall(r"[-+]?\d*\.\d+|\d+", target_line)
            if not nums:
                continue

            vals.append(float(nums[-1]))

        # print(vals)
        if vals:
            avg = float(np.mean(vals))
            sd  = float(np.std(vals, ddof=0))
            results[folder.name] = (avg, sd)

    # ─── Pick the “best” folder ────────────────────────────────────────────────────
    if not results:
        # print("No results found for the given setting.")
        return None, None, None

    if dataset_name in LOWER_IS_BETTER:
        best_folder, (best_avg, best_sd) = min(results.items(), key=lambda kv: kv[1][0])
    else:
        best_folder, (best_avg, best_sd) = max(results.items(), key=lambda kv: kv[1][0])

    # ─── Report ───────────────────────────────────────────────────────────────────
    # print("\nPer-folder performance:")
    # for name, (avg, sd) in results.items():
    #     print(f"  • {name:50s} → mean={avg:.4f}, std={sd:.4f}")

    # print("\nBest hyperparameter setting:")
    # print(f"  → {best_folder}")
    # print(f"     Mean = {best_avg:.4f}, Std = {best_sd:.4f}")
    return best_folder, best_avg, best_sd

In [21]:
dataset_name = 'cep'
fewshot_bool = True
fewshot_num = 500
split = 'scaffold'
base_loss = 'mse' if dataset_name in ['lipo', 'esol', 'malaria', 'cep'] else 'filter_bce'
loss_funcs = [base_loss, base_loss+"+surg", base_loss+"+lpft", base_loss+"+l2_sp", base_loss+"+feature_map", base_loss+"+bss"]
# full_or_LP = ["fullFT", "LP", "Lora"]
full_or_LP = ['Lora']
# for split in splits:
for loss_fun in loss_funcs:
    for full_bool in full_or_LP:
        best_folder, best_avg, best_sd = calc_score(dataset_name, split, fewshot_bool, fewshot_num, full_bool, loss_fun)
        if best_folder:
            print("\nBest hyperparameter setting:")
            print(f"  → {best_folder}")
            if dataset_name not in ['lipo', 'esol', 'malaria', 'cep']:
                # percentage form, ready to paste into one Google-Sheets cell
                pct_mean = best_avg * 100
                pct_std  = best_sd  * 100
                formatted = f"{pct_mean:.2f} ± {pct_std:.2f}"
            else:
                # plain decimal with three digits
                formatted = f"{best_avg:.3f} ± {best_sd:.3f}"
            print(f"     {formatted}")


Best hyperparameter setting:
  → FT_cep_scaffold_Fewshot_True_500_Lora_type_mse_m0.001_h0.001
     1.569 ± 0.040
